# 03 — Ranking by Seed

Production ranking on the committed fixture. Scoring is cosine similarity of
FeatureVectors after the Catalog StandardScaler. The Seed itself is omitted.
Several Seeds: each candidate's score is the **maximum** similarity to any
Seed, not the mean.

**Here:** FeatureVectors, one Seed, two dissimilar Seeds (max-not-mean),
unknown Seed failure, DriveTime as a stubbed query-time filter.

**Not here:** ingest mechanics (`01_raw_data_and_ingestion.ipynb`), Catalog
maps (`02_catalog.ipynb`), adding columns (`04_rank_feature_lab.ipynb`).

This notebook does not download Geofabrik extracts and does not call a live
Valhalla instance.


In [ ]:
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

from dayhike import build_catalog

catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
catalog = build_catalog(fixture, catalog_path)
print(catalog.ingest_report)
print("sqlite", catalog.path)
print("attribution", catalog.attribution)
for hike in catalog.dayhikes:
    print(hike.name)
    print(hike.feature_vector)


## FeatureVector

These columns are what Ranking sees. DriveTime is not one of them. Missing
elevation or `sac_scale` is NaN in the matrix (filled with 0 after scaling),
not a silent zero in SQLite.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)
import numpy as np
import pandas as pd

from dayhike import feature_matrix
from dayhike.catalog import FEATURE_NAMES

matrix = feature_matrix(catalog.dayhikes)
frame = pd.DataFrame(matrix, columns=FEATURE_NAMES, index=[hike.name for hike in catalog.dayhikes])
print(frame.T.to_string())
print("scaler")
for name, mean, scale in zip(
    catalog.scaler.feature_names, catalog.scaler.mean, catalog.scaler.scale
):
    print(f"  {name}\tmean={mean:.4g}\tscale={scale:.4g}")

fig, ax = plt.subplots(figsize=(8, 4))
filled = np.where(np.isnan(matrix), 0.0, matrix)
# column-wise min-max so mixed units are visible together
span = filled.max(axis=0) - filled.min(axis=0)
span[span == 0] = 1.0
normed = (filled - filled.min(axis=0)) / span
im = ax.imshow(normed, aspect="auto", cmap="YlGn")
ax.set_yticks(range(len(catalog.dayhikes)))
ax.set_yticklabels([hike.name for hike in catalog.dayhikes])
ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=60, ha="right", fontsize=8)
ax.set_title("FeatureVector (min-max per column, NaN→0)")
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout()
show_fig(fig)


## Ranking by Seed

One Seed omits itself and ranks the rest. Two dissimilar Seeds score each
candidate by **maximum** similarity to any Seed, not the mean — otherwise a
ridge and a valley would collapse into a mushy middle. The max-not-mean overlay
adds two extra candidates on top of this Catalog.


In [ ]:
from dataclasses import replace

from dayhike import rank_dayhikes

for item in rank_dayhikes(catalog, ["Ridge Walk"]).items:
    print(f"one_seed\t{item.dayhike.name}\t{item.score}")

for item in rank_dayhikes(catalog, ["Valley Path"]).items:
    print(f"one_seed_valley\t{item.dayhike.name}\t{item.score}")

ridge = next(hike for hike in catalog.dayhikes if hike.id == 1001)
valley = next(hike for hike in catalog.dayhikes if hike.id == 1002)
near_ridge = replace(ridge, id=9001, name="Near Ridge")
midpoint = replace(
    valley,
    id=9002,
    name="Mid Path",
    length_m=(ridge.length_m + valley.length_m) / 2,
    feature_vector=replace(
        valley.feature_vector,
        length_m=(ridge.feature_vector.length_m + valley.feature_vector.length_m) / 2,
        elevation_min_m=ridge.feature_vector.elevation_min_m,
        elevation_max_m=ridge.feature_vector.elevation_max_m,
        elevation_gain_m=ridge.feature_vector.elevation_gain_m,
        difficulty_ordinal=1,
    ),
)
extended = replace(catalog, dayhikes=(ridge, valley, near_ridge, midpoint))
only_ridge = {
    item.dayhike.name: item.score
    for item in rank_dayhikes(extended, ["Ridge Walk"]).items
}
only_valley = {
    item.dayhike.name: item.score
    for item in rank_dayhikes(extended, ["Valley Path"]).items
}
both = rank_dayhikes(extended, ["Ridge Walk", "Valley Path"])
for item in both.items:
    print(f"two_seeds\t{item.dayhike.name}\t{item.score}")
for name in ("Near Ridge", "Mid Path"):
    score = next(item.score for item in both.items if item.dayhike.name == name)
    expected_max = max(only_ridge[name], only_valley[name])
    expected_mean = (only_ridge[name] + only_valley[name]) / 2
    print(f"max_not_mean\t{name}\t{score}\t{expected_max}\t{expected_mean}")


In [ ]:
ridge_rank = rank_dayhikes(catalog, ["Ridge Walk"])
fig, ax = plt.subplots(figsize=(7, 3.5))
names = [item.dayhike.name for item in ridge_rank.items]
scores = [item.score for item in ridge_rank.items]
ax.barh(names[::-1], scores[::-1], color="#1d4ed8")
ax.set_xlabel("cosine similarity")
ax.set_title("Neighbours of Ridge Walk")
fig.tight_layout()
show_fig(fig)


## Unknown Seed

A Seed name that is not in the Catalog fails loudly. There is no fuzzy lookup.


In [ ]:
from dayhike import UnknownSeedName, rank_dayhikes

try:
    rank_dayhikes(catalog, ["Unknown Ridge"])
except UnknownSeedName as exc:
    print(type(exc).__name__)
    print(exc)


## DriveTime filter (stub client)

DriveTime is a query-time filter from today's Origin, not a FeatureVector
column. This cell stubs the Valhalla matrix client so CI never needs routing
tiles. Real routing from a live Origin needs local Valhalla tiles.

Valhalla is **not** a pip package and is **not** started by this notebook. It
is a Docker service (`data/valhalla/compose.yaml`) that listens on
`http://127.0.0.1:8002` after `prepare-roads.sh` writes a car-only PBF into
`data/valhalla/` and you run compose. `dayhike` then POSTs `/sources_to_targets`.
Ranking without Origin never connects.


In [ ]:
from dayhike import Origin, filter_by_drive_time, rank_dayhikes


class StubMatrix:
    def __init__(self, time_s):
        self._time_s = time_s

    def sources_to_targets(self, origin, trailheads):
        return [self._time_s] * len(trailheads)


origin = Origin(latitude=45.8, longitude=15.97)
ranking = rank_dayhikes(catalog, ["Ridge Walk"])
kept = filter_by_drive_time(ranking, origin, 2, StubMatrix(1800.0))
for item in kept.items:
    print(f"drive_keep\t{item.dayhike.name}\t{item.drive_time.hours}")
dropped = filter_by_drive_time(ranking, origin, 2, StubMatrix(None))
print(f"drive_drop_count\t{len(dropped.items)}")


## Limitations

- The Catalog unit is `route=hiking` relations in a length band (1–20 km). Unnamed `highway=path` ways never become DayHikes; neither do superroutes or out-of-band lengths. Blank OSM `name` is kept with a Seed label and `name_origin`.
- DriveTime for a real Origin needs local Valhalla tiles; the stub above is only a stand-in.
- There are no scraped reviews and no webpage ratings yet. Ratings and UI are future work.
- Ingest reads OSM XML or PBF (pyosmium). `dayhike fetch-extracts` downloads the Geofabrik list.
- AmenityFacts count nodes, ways, and tagged relations within 20 km of the Trailhead.

**Next:** [`04_rank_feature_lab.ipynb`](04_rank_feature_lab.ipynb) is the
sandbox for trying new FeatureVector columns against this same scorer.
